In [1]:
import pandas as pd
import numpy as np
import torch
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import jax
import jax.numpy as jnp
import optax
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

/Users/art/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/art/anaconda3/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
data = fetch_california_housing(as_frame = True)
df = data.frame

In [3]:
df.shape

(20640, 9)

**Every framework example follows the same five steps:**

1. Represent the data and parameters as tensors/arrays
2. Forward pass — compute predictions
3. Loss — measure how wrong the predictions are (Mean Squared Error)
4. Backward pass — compute gradients of the loss w.r.t. the parameters
5. Update — nudge the parameters using the gradients (gradient descent)

In [4]:
df.columns

Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude', 'MedHouseVal'],
      dtype='object')

In [27]:
X = df.drop(columns=['MedHouseVal', 'AveBedrms','Population']).values.astype(np.float32)
y = df['MedHouseVal'].values.astype(np.float32).reshape(-1,1)

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X,y, random_state=42)

X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0)
X_train_s = (X_train - X_mean)/X_std
X_test_s = (X_test - X_mean)/ X_std

In [7]:
X_train_t = torch.from_numpy(X_train_s)
y_train_t = torch.from_numpy(y_train)
X_test_t = torch.from_numpy(X_test_s)
y_test_t = torch.from_numpy(y_test)

In [8]:
torch.manual_seed(42)
n_features = X_train_s.shape[1]
n_features

6

In [9]:
# help(torch.nn.Linear)

In [10]:
model = torch.nn.Linear(in_features=n_features, out_features=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = torch.nn.MSELoss()


epochs = 200
for epoch in range(epochs):
    y_pred = model(X_train_t)
    loss = loss_fn(y_pred, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 40 == 0:
        print(f"epoch {epoch:4d} | train MSE {loss.item():.4f}")
        
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_t)
    test_mse = loss_fn(y_test_pred, y_test_t).item() 
    ss_res = torch.sum((y_test_t-y_test_pred)**2)
    ss_tot = torch.sum((y_test_t - y_test_t.mean())**2)
    test_r2 = (1 - ss_res / ss_tot).item()
print(f"MSE {test_mse:.4}")
print(f"R2 {test_r2:.4}")

epoch    0 | train MSE 6.3031
epoch   40 | train MSE 0.8348
epoch   80 | train MSE 0.5390
epoch  120 | train MSE 0.5384
epoch  160 | train MSE 0.5384
MSE 0.5358
R2 0.5951


# TensorFlow Solution

In [21]:
X_tf = tf.constant(X_train_s)
y_tf = tf.constant(y_train)
X_test_tf = tf.constant(X_test_s)
y_test_tf = tf.constant(y_test)

tf.random.set_seed(42)
n_features = X_train_s.shape[1]


model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=(n_features,))
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.05),
    loss = "mse"
)
history = model.fit(X_tf, y_tf, epochs=200, verbose=0)

test_mse = model.evaluate(X_test_tf, y_test_tf, verbose=0)
y_test_pred = model.predict(X_test_tf, verbose=0)
ss_res = np.sum((y_test - y_test_pred)**2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
test_r2 = 1 - ss_res / ss_tot
print(f"MSE {test_mse:.4}")
print(f"R2 {test_r2:.4}")

/Users/art/anaconda3/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


MSE 0.5487
R2 0.5854


In [20]:
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 1)              │             7 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23 (96.00 B)

 Trainable params: 7 (28.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 16 (68.00 B)

# Jax Solution

In [37]:
x_jax = jnp.array(X_train_s)
y_jax = jnp.array(y_train)
x_test_jax = jnp.array(X_test_s)
y_test_jax = jnp.array(y_test)

def predict(params, x):
    w,b = params
    return x @ w + b

def loss_fn(params, x,y):
    preds = predict(params, x)
    return jnp.mean((preds - y)**2)

loss_and_grad_fn = jax.jit(jax.value_and_grad(loss_fn))

key = jax.random.PRNGKey(42)
w_key, b_key = jax.random.split(key)
n_features = X_train_s.shape[1]
params = (jax.random.normal(w_key, (n_features,1)) * 0.05, jnp.zeros((1,)))

optimizer = optax.adam(learning_rate=0.05)
opt_state = optimizer.init(params)

epochs = 200

for epoch in range(epochs):
    loss, grads = loss_and_grad_fn(params, x_jax, y_jax)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    if epoch%40 == 0:
        print(f"epoch {epoch}   MSE {loss:.4f}")

test_pred = predict(params, x_test_jax)
test_mse = jnp.mean((test_pred - y_test_jax) ** 2)
ss_res = jnp.sum((y_test_jax - test_pred)**2)
ss_tot = jnp.sum((y_test_jax - y_test_jax.mean()) ** 2)
test_r2 = 1 - ss_res / ss_tot
print(f"MSE {test_mse:.4}")
print(f"R2 {test_r2:.4}")

epoch 0   MSE 5.6113
epoch 40   MSE 0.6850
epoch 80   MSE 0.5403
epoch 120   MSE 0.5384
epoch 160   MSE 0.5384
MSE 0.5358
R2 0.5951


In [33]:
a = jnp.array([1,2,3])
b = jnp.array([4,5,6])
jnp.dot(a,b)

Array(32, dtype=int32)

In [35]:
(1 * 4) + (2 * 5) + (3 * 6) 

32

In [36]:
a_np = np.array([1,2,3])
b_np = np.array([4,5,6])
np.dot(a,b)

32